# Notebook 02 — Data Cleaning & Advanced Imputation

**Author:** Section D – Group 8  
**Input:** `data/extracted_subset_80k.csv`  
**Output:** `data/clean_data.csv` + `reports/cleaning_report.csv`  

**Strategy:**
- Filter LOT listings (land only) to focus on residential buildings
- Convert acres → sqft before dropping unit column
- Drop 12 useless/metadata columns
- Treat 0 values in bedrooms/bathrooms/space as missing (not real)
- 3-Tier Hierarchical Imputation for `living_space` and `bedroom_number`
- Bathroom imputation from space ratios
- Address parsing via regex to recover missing city/street
- Geolocation recovery (postcode → city → state centroids)
- Clamp bedrooms/bathrooms to [1, 10]
- Outlier capping at 99th percentile
- All steps logged to `cleaning_report.csv`

## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import re
import os

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.2f}'.format)

INPUT_PATH  = '../data/extracted_subset_80k.csv'
OUTPUT_PATH = '../data/clean_data.csv'
LOG_PATH    = '../reports/cleaning_report.csv'

cleaning_log = []

def log(step, details, count):
    cleaning_log.append({'Step': step, 'Details': details, 'RowsAffected': int(count)})
    print(f'[✓] {step}: {details} → {count} rows')

print('Setup complete.')

Setup complete.


## 1. Load the 80k Sample

In [2]:
df = pd.read_csv(INPUT_PATH, low_memory=False)
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')

Loaded: 80,000 rows × 28 columns


In [3]:
# Snapshot of raw nulls before any cleaning
print('--- RAW Null Counts (Before Cleaning) ---')
raw_nulls = df.isnull().sum()
print(raw_nulls[raw_nulls > 0].to_string())

--- RAW Null Counts (Before Cleaning) ---
street_name           22
apartment          78003
state                  1
latitude            9415
longitude           9415
postcode               2
bedroom_number     20563
bathroom_number    16952
price_per_unit     21701
living_space       20064
land_space         11321
land_space_unit    11321
broker_id          80000
year_build         80000
total_num_units    80000
agency_name        20771
agent_name         80000
agent_phone        80000


In [4]:
# Raw property type counts
print('Property types in raw sample:')
print(df['property_type'].value_counts())

Property types in raw sample:
property_type
SINGLE_FAMILY    47504
LOT              20418
CONDO             4927
TOWNHOUSE         2704
MANUFACTURED      2230
MULTI_FAMILY      2164
APARTMENT           53
Name: count, dtype: int64


## 2. Filter: Residential Buildings Only

Remove `LOT` (land-only) rows since they have no rooms or living space — keeping them would create a massive artificial spike in missing values for all housing metrics.

In [5]:
before = len(df)
df = df[df['property_type'] != 'LOT'].copy()
after = len(df)
log('Filter LOT', 'Removed land-only listings', before - after)
print(f'Remaining rows: {after:,}')

[✓] Filter LOT: Removed land-only listings → 20418 rows
Remaining rows: 59,582


In [6]:
# Confirm property types after filter
print('Residential property type breakdown:')
print(df['property_type'].value_counts())

Residential property type breakdown:
property_type
SINGLE_FAMILY    47504
CONDO             4927
TOWNHOUSE         2704
MANUFACTURED      2230
MULTI_FAMILY      2164
APARTMENT           53
Name: count, dtype: int64


## 3. Acres → SqFt Conversion

Must happen BEFORE dropping `land_space_unit`.

In [7]:
df['land_space'] = pd.to_numeric(df['land_space'], errors='coerce')
if 'land_space_unit' in df.columns:
    acres_mask = df['land_space_unit'] == 'acres'
    df.loc[acres_mask, 'land_space'] = df.loc[acres_mask, 'land_space'] * 43560
    log('Acres→SqFt', 'Converted acres to sqft (×43,560)', int(acres_mask.sum()))
    print(f'Rows converted: {int(acres_mask.sum()):,}')

[✓] Acres→SqFt: Converted acres to sqft (×43,560) → 19090 rows
Rows converted: 19,090


## 4. Drop Useless Columns

In [8]:
# Check null % for all columns
null_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print('Top columns by null %:')
print(null_pct[null_pct > 0].round(2))

Top columns by null %:
total_num_units   100.00
agent_phone       100.00
broker_id         100.00
agent_name        100.00
year_build        100.00
apartment          96.66
agency_name        26.65
land_space_unit    17.94
land_space         17.94
latitude            6.65
longitude           6.65
price_per_unit      2.88
living_space        1.88
bedroom_number      1.25
bathroom_number     0.68
street_name         0.00
postcode            0.00
dtype: float64


In [9]:
DROP_COLS = [
    'apartment', 'broker_id', 'year_build', 'total_num_units',
    'agent_name', 'agent_phone', 'is_owned_by_zillow',
    'price_per_unit', 'land_space_unit',
    'property_url',    # URL not needed for analysis
    'listing_age',     # All values are -1 (no useful data)
    'RunDate',         # Single scrape date for all rows
]
existing_drop = [c for c in DROP_COLS if c in df.columns]
df.drop(columns=existing_drop, inplace=True)
log('Drop Columns', f'Removed {len(existing_drop)} columns', len(df))
print(f'Remaining columns ({df.shape[1]}): {list(df.columns)}')

[✓] Drop Columns: Removed 12 columns → 59582 rows
Remaining columns (16): ['property_id', 'address', 'street_name', 'city', 'state', 'latitude', 'longitude', 'postcode', 'price', 'bedroom_number', 'bathroom_number', 'living_space', 'land_space', 'property_type', 'property_status', 'agency_name']


## 5. Type Conversion

In [10]:
num_cols = ['price', 'living_space', 'bedroom_number', 'bathroom_number',
            'latitude', 'longitude', 'land_space']
for col in num_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
print('Numeric conversion done.')

Numeric conversion done.


## 6. Zero → NaN Conversion

A house with 0 sqft or 0 bedrooms is not real data — these are missing values disguised as zero.

In [11]:
for col in ['living_space', 'bedroom_number', 'bathroom_number']:
    count = (df[col] == 0).sum()
    df.loc[df[col] == 0, col] = np.nan
    log('Zero→NaN', f'{col}: treated {count} zeros as missing', count)

[✓] Zero→NaN: living_space: treated 592 zeros as missing → 592 rows
[✓] Zero→NaN: bedroom_number: treated 452 zeros as missing → 452 rows
[✓] Zero→NaN: bathroom_number: treated 581 zeros as missing → 581 rows


## 7. Address Recovery via Regex

Pattern: `street, city, STATE postcode` — we extract missing city/street from the raw address string.

In [12]:
print(f'Null street_name before: {df["street_name"].isnull().sum()}')
print(f'Null city before       : {df["city"].isnull().sum()}')

Null street_name before: 1
Null city before       : 0


In [13]:
def recover_address(row):
    if pd.isna(row['street_name']) or pd.isna(row['city']):
        m = re.match(r'^([^,]+),\s*([^,]+),\s*([A-Z]{2})\s*(\d{5})', str(row['address']))
        if m:
            if pd.isna(row['street_name']):
                row['street_name'] = m.group(1).strip()
            if pd.isna(row['city']):
                row['city'] = m.group(2).strip()
    return row

df = df.apply(recover_address, axis=1)
df['street_name'] = df['street_name'].fillna('Unknown Street')
df['city']        = df['city'].fillna('Unknown City')

# Drop rows with null postcode (needed for geolocation groupby)
null_post = df['postcode'].isnull().sum()
df = df.dropna(subset=['postcode'])
log('Address Recovery', f'Regex + Unknown fallback; dropped {null_post} null-postcode rows', df.shape[0])
print(f'Null street_name: {df["street_name"].isnull().sum()}')
print(f'Null city       : {df["city"].isnull().sum()}')
print(f'Null postcode   : {df["postcode"].isnull().sum()}')

[✓] Address Recovery: Regex + Unknown fallback; dropped 1 null-postcode rows → 59581 rows
Null street_name: 0
Null city       : 0
Null postcode   : 0


## 8. Derive Imputation Ratios

In [14]:
df['sqft_per_room'] = df['living_space'] / df['bedroom_number']
city_sqft_room = df.groupby('city')['sqft_per_room'].transform('median')
nat_sqft_room  = df['sqft_per_room'].median()
print(f'National median sqft/bedroom : {nat_sqft_room:.0f}')

National median sqft/bedroom : 591


In [15]:
df['pps'] = df['price'] / df['living_space']
city_pps = df.groupby('city')['pps'].transform('median')
nat_pps  = df['pps'].median()
print(f'National median price/sqft: ${nat_pps:.2f}')

National median price/sqft: $221.67


## 9. Tier 1 Imputation — Living Space from Bedrooms

**Formula:** `living_space = bedroom_number × city_median(sqft/bedroom)`

In [16]:
initial = df['living_space'].isnull().sum()
print(f'Null living_space BEFORE Tier 1: {initial}')

mask_t1 = df['living_space'].isnull() & df['bedroom_number'].notnull()
df.loc[mask_t1, 'living_space'] = (
    df.loc[mask_t1, 'bedroom_number'] * city_sqft_room.fillna(nat_sqft_room)
)

filled = initial - df['living_space'].isnull().sum()
log('Tier 1: Bed→Space', f'bedroom × city_median(sqft/bed)', filled)
print(f'Null living_space AFTER Tier 1 : {df["living_space"].isnull().sum()}')

Null living_space BEFORE Tier 1: 1714
[✓] Tier 1: Bed→Space: bedroom × city_median(sqft/bed) → 1126 rows
Null living_space AFTER Tier 1 : 588


## 10. Tier 2 Imputation — Living Space from Price

**Formula:** `living_space = price ÷ city_median(price/sqft)`

In [17]:
initial2 = df['living_space'].isnull().sum()
print(f'Null living_space BEFORE Tier 2: {initial2}')

mask_t2 = df['living_space'].isnull()
denom = city_pps.fillna(nat_pps).replace(0, nat_pps)  # guard div-by-zero
df.loc[mask_t2, 'living_space'] = df.loc[mask_t2, 'price'] / denom

filled2 = initial2 - df['living_space'].isnull().sum()
log('Tier 2: Price→Space', f'price ÷ city_median(price/sqft)', filled2)
print(f'Null living_space AFTER Tier 2 : {df["living_space"].isnull().sum()}')

Null living_space BEFORE Tier 2: 588
[✓] Tier 2: Price→Space: price ÷ city_median(price/sqft) → 588 rows
Null living_space AFTER Tier 2 : 0


## 11. Tier 3 — Bedrooms & Bathrooms from Space

**Bedrooms:** `round(living_space ÷ city_median(sqft/bedroom))`  
**Bathrooms:** `round(living_space ÷ city_median(sqft/bathroom))`

In [18]:
# Bedrooms
initial_beds = df['bedroom_number'].isnull().sum()
df['bedroom_number'] = df['bedroom_number'].fillna(
    (df['living_space'] / city_sqft_room.fillna(nat_sqft_room)).round()
)
log('Tier 3: Space→Bed', 'Reversed from living_space', initial_beds - df['bedroom_number'].isnull().sum())
print(f'Null bedroom_number remaining: {df["bedroom_number"].isnull().sum()}')

[✓] Tier 3: Space→Bed: Reversed from living_space → 1195 rows
Null bedroom_number remaining: 0


In [19]:
# Bathrooms
df['sqft_per_bath'] = df['living_space'] / df['bathroom_number']
city_sqft_bath = df.groupby('city')['sqft_per_bath'].transform('median')
nat_sqft_bath  = df['sqft_per_bath'].median()

initial_baths = df['bathroom_number'].isnull().sum()
df['bathroom_number'] = df['bathroom_number'].fillna(
    (df['living_space'] / city_sqft_bath.fillna(nat_sqft_bath)).round()
)
log('Bathrooms Impute', 'living_space ÷ city median sqft/bath', initial_baths - df['bathroom_number'].isnull().sum())
print(f'Null bathroom_number remaining: {df["bathroom_number"].isnull().sum()}')

[✓] Bathrooms Impute: living_space ÷ city median sqft/bath → 987 rows
Null bathroom_number remaining: 0


## 12. Clamp Bedrooms & Bathrooms to [1, 10]

A real residential listing has at least 1 bed and 1 bath. Listings claiming 3,741 bedrooms are multi-unit commercial data errors.

In [20]:
print(f'Before clamp — Bed range: {df["bedroom_number"].min():.0f}–{df["bedroom_number"].max():.0f}')
print(f'Before clamp — Bath range: {df["bathroom_number"].min():.0f}–{df["bathroom_number"].max():.0f}')

df['bedroom_number']  = df['bedroom_number'].clip(lower=1, upper=10)
df['bathroom_number'] = df['bathroom_number'].clip(lower=1, upper=10)
log('Clamp Bed/Bath', 'Clipped to [1, 10]', len(df))

print(f'After clamp  — Bed range: {df["bedroom_number"].min():.0f}–{df["bedroom_number"].max():.0f}')
print(f'After clamp  — Bath range: {df["bathroom_number"].min():.0f}–{df["bathroom_number"].max():.0f}')

Before clamp — Bed range: 0–3741
Before clamp — Bath range: 0–1892
[✓] Clamp Bed/Bath: Clipped to [1, 10] → 59581 rows
After clamp  — Bed range: 1–10
After clamp  — Bath range: 1–10


## 13. Geolocation Imputation

In [21]:
print(f'Null latitude BEFORE : {df["latitude"].isnull().sum()}')

for col in ['latitude', 'longitude']:
    df[col] = (
        df[col]
        .fillna(df.groupby('postcode')[col].transform('median'))
        .fillna(df.groupby('city')[col].transform('median'))
        .fillna(df.groupby('state')[col].transform('median'))
    )

log('Geolocation', 'postcode → city → state centroid fallback', df.shape[0])
print(f'Null latitude AFTER  : {df["latitude"].isnull().sum()}')

Null latitude BEFORE : 3960
[✓] Geolocation: postcode → city → state centroid fallback → 59581 rows
Null latitude AFTER  : 0


## 14. Land Space — Median Fill + Fix Negatives

In [22]:
initial_land = df['land_space'].isnull().sum()
df['land_space'] = (
    df['land_space']
    .fillna(df.groupby('city')['land_space'].transform('median'))
    .fillna(df.groupby('state')['land_space'].transform('median'))
)

# Fix negative values
neg_count = (df['land_space'] < 0).sum()
df.loc[df['land_space'] < 0, 'land_space'] = df.groupby('city')['land_space'].transform('median')
log('Land Space', f'Filled {initial_land} nulls + fixed {neg_count} negatives', initial_land + neg_count)
print(f'Remaining null land_space: {df["land_space"].isnull().sum()}')

[✓] Land Space: Filled 10690 nulls + fixed 9 negatives → 10699 rows
Remaining null land_space: 0


## 15. Agency Name + Final Row Filter

In [23]:
df['agency_name'] = df['agency_name'].fillna('Unknown Agency')
log('Agency Name', 'Filled nulls with Unknown Agency', df.shape[0])

[✓] Agency Name: Filled nulls with Unknown Agency → 59581 rows


In [24]:
# Remove any remaining rows with 0/inf/null in critical features
before_final = len(df)
df = df[
    df['living_space'].notna() &
    ~np.isinf(df['living_space']) &
    (df['living_space'] > 0) &
    df['bedroom_number'].notna() &
    df['bathroom_number'].notna() &
    df['price'].notna() &
    (df['price'] > 0)
].copy()
log('Final Row Filter', 'Removed rows with 0/inf/null in critical features', before_final - len(df))
print(f'Rows remaining: {len(df):,}')

[✓] Final Row Filter: Removed rows with 0/inf/null in critical features → 0 rows
Rows remaining: 59,581


## 16. Outlier Capping & Derived Columns

In [25]:
price_99 = df['price'].quantile(0.99)
space_99 = df['living_space'].quantile(0.99)
print(f'99th pct cap — Price       : ${price_99:,.0f}')
print(f'99th pct cap — Living Space: {space_99:,.0f} sqft')

df['Sale_Price']         = df['price']
df['Sale_Price_Capped']  = df['price'].clip(upper=price_99)
df['Living_Space_Capped'] = df['living_space'].clip(upper=space_99)
df['Price_Per_SqFt']     = df['price'] / df['living_space']
log('Derived Columns', f'Sale_Price, Capped, Price_Per_SqFt', len(df))

99th pct cap — Price       : $4,796,880
99th pct cap — Living Space: 7,369 sqft
[✓] Derived Columns: Sale_Price, Capped, Price_Per_SqFt → 59581 rows


In [26]:
# Drop temp columns and rename
DROP_TEMP = ['pps', 'sqft_per_room', 'sqft_per_bath', 'price']
df.drop(columns=[c for c in DROP_TEMP if c in df.columns], inplace=True)

df.rename(columns={
    'bedroom_number':  'Bedrooms',
    'bathroom_number': 'Bathrooms',
    'living_space':    'Living_Space_SqFt'
}, inplace=True)
print(f'Final columns ({df.shape[1]}): {list(df.columns)}')

Final columns (19): ['property_id', 'address', 'street_name', 'city', 'state', 'latitude', 'longitude', 'postcode', 'Bedrooms', 'Bathrooms', 'Living_Space_SqFt', 'land_space', 'property_type', 'property_status', 'agency_name', 'Sale_Price', 'Sale_Price_Capped', 'Living_Space_Capped', 'Price_Per_SqFt']


## 17. Final Verification

In [27]:
print('=== FINAL NULL COUNTS ===')
nulls = df.isnull().sum()
print(nulls)
print(f'\nTotal nulls: {nulls.sum()}')

=== FINAL NULL COUNTS ===
property_id            0
address                0
street_name            0
city                   0
state                  0
latitude               0
longitude              0
postcode               0
Bedrooms               0
Bathrooms              0
Living_Space_SqFt      0
land_space             0
property_type          0
property_status        0
agency_name            0
Sale_Price             0
Sale_Price_Capped      0
Living_Space_Capped    0
Price_Per_SqFt         0
dtype: int64

Total nulls: 0


In [28]:
# Check for hidden issues: zeros + infinities
critical = ['Sale_Price', 'Living_Space_SqFt', 'Bedrooms', 'Bathrooms', 'Price_Per_SqFt']
for col in critical:
    n_null = df[col].isnull().sum()
    n_zero = (df[col] == 0).sum()
    n_inf  = np.isinf(df[col]).sum()
    status = '✅' if (n_null + n_zero + n_inf == 0) else '❌'
    print(f'{col:25}: nulls={n_null}  zeros={n_zero}  inf={n_inf}  {status}')

Sale_Price               : nulls=0  zeros=0  inf=0  ✅
Living_Space_SqFt        : nulls=0  zeros=0  inf=0  ✅
Bedrooms                 : nulls=0  zeros=0  inf=0  ✅
Bathrooms                : nulls=0  zeros=0  inf=0  ✅
Price_Per_SqFt           : nulls=0  zeros=0  inf=0  ✅


In [29]:
# Final shape and summary
print(f'Final dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')
print()
df[['Sale_Price', 'Living_Space_SqFt', 'Bedrooms', 'Bathrooms']].describe().round(2)

Final dataset: 59,581 rows × 19 columns



,Sale_Price,Living_Space_SqFt,Bedrooms,Bathrooms
count,59581.00,59581.00,59581.00,59581.00
mean,689513.78,2245.49,3.40,2.65
std,1697173.19,6375.45,1.23,1.25
min,1.00,1.00,1.00,1.00
25%,279000.00,1403.00,3.00,2.00
50%,439990.00,1900.00,3.00,2.00
75%,695000.00,2607.00,4.00,3.00
max,160000000.00,1454468.00,10.00,10.00


## 18. Save Clean Data & Log

In [30]:
df.to_csv(OUTPUT_PATH, index=False)
pd.DataFrame(cleaning_log).to_csv(LOG_PATH, index=False)
print(f'Clean data saved to : {OUTPUT_PATH}')
print(f'Cleaning log saved  : {LOG_PATH}')
print(f'Final rows          : {len(df):,}')

Clean data saved to : ../data/clean_data.csv
Cleaning log saved  : ../reports/cleaning_report.csv
Final rows          : 59,581
